# RAAF Ablation: Unbalanced Pooling
### Resolving the backbone-undertraining confound

The main RAAF run (raaf_architecture.ipynb) used **register-balanced pooling**: HARD's
training split was down-sampled from ~75K to ~11.4K examples (an 88% cut) to match
ASTD+ArSAS's combined size, per the Kumar et al. (2022) feature-distortion concern.

**The confound this created**: RAAF underperformed all baselines on all three datasets
(HARD 95.60 vs. CAMeLBERT 96.24; ASTD 87.09 vs. MARBERT 87.62; ArSAS 90.86 vs. MARBERT
91.24) -- but its backbones saw far less training data than the baseline backbones did.
We cannot currently distinguish "the fusion mechanism fails" from "the backbones are
undertrained."

**This ablation isolates that confound**: identical pipeline, but with
`balance_by_register=False` (full pooled data, ~86K train samples). All checkpoints,
embeddings, and results go to a **separate `raaf_unbalanced/` directory** -- nothing
from the balanced run is touched or overwritten, so the two runs remain directly
comparable at the end (Section 15).

**TIME WARNING**: unbalanced pooled training is ~3.8x more data per backbone than
the balanced run. Based on the balanced run's measured 12.5 min/model/seed on T4,
expect roughly **45-50 min per model per seed, so ~16 hours total for Stage 1**
(4 backbones x 5 seeds). Fully checkpointed -- run it across as many sessions as
needed; completed work is never repeated. If that's too much, reducing SEEDS to
[42, 123, 456] in Section 2 cuts it to ~9.5h at the cost of weaker statistics
(note this deviation in the paper if used).

Interpretation guide for the final comparison (Section 15):
- **Unbalanced RAAF competitive with baselines**: the balanced run's shortfall was the
  undertraining confound, and register-balanced pooling was too aggressive. The fusion
  mechanism itself is not refuted.
- **Unbalanced RAAF still clearly below baselines**: the shortfall is not mainly the
  data cut; the evidence turns against the fusion mechanism providing accuracy value,
  and the paper should report RAAF as mechanism-level findings (P1-P3) with an
  honestly negative accuracy result.


## 1. Setup

In [1]:
!pip install -q transformers datasets torch scikit-learn accelerate


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
EL4ASA_ROOT = "/content/drive/MyDrive/EL4ASA"
RAAF_DIR = f"{EL4ASA_ROOT}/raaf_unbalanced"   # SEPARATE tree -- balanced run untouched
BACKBONE_CKPT_DIR = f"{RAAF_DIR}/backbones"
RAAF_RESULTS_DIR = f"{RAAF_DIR}/results"
os.makedirs(BACKBONE_CKPT_DIR, exist_ok=True)
os.makedirs(RAAF_RESULTS_DIR, exist_ok=True)
print(f"ABLATION RUN -- outputs isolated under: {RAAF_DIR}")


Mounted at /content/drive
ABLATION RUN -- outputs isolated under: /content/drive/MyDrive/EL4ASA/raaf_unbalanced


In [3]:
import re
import json
import time
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected -- go to Runtime > Change runtime type > T4 GPU.")


Device: cuda


## 2. Configuration

In [4]:
MODELS = {
    "arabert": "aubmindlab/bert-base-arabert",
    "marbert": "UBC-NLP/MARBERT",
    "xlm-roberta": "xlm-roberta-base",
    "camelbert": "CAMeL-Lab/bert-base-arabic-camelbert-msa",
}
MODEL_NAMES = list(MODELS.keys())

SEEDS = [42, 123, 456, 789, 2024]   # reduce if the timing test says to
LEARNING_RATE = 2e-5
RAAF_LEARNING_RATE = 1e-3   # RAAF module (small, on frozen features) can use a higher LR
BATCH_SIZE = 16
NUM_EPOCHS_BACKBONE = 3     # joint backbone fine-tuning
NUM_EPOCHS_RAAF = 10        # RAAF module training on cached frozen embeddings (cheap, more epochs OK)
MAX_LENGTH = 128
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1
NUM_SENTIMENT_LABELS = 2
NUM_REGISTER_LABELS = 2     # 0=formal (HARD), 1=informal (ASTD/ArSAS)
REGISTER_LOSS_WEIGHT = 0.3  # weight of auxiliary register loss relative to main sentiment loss
FUSION_DIM = 256            # dimensionality of the cross-attention fusion space
NUM_FUSION_HEADS = 4

print("Config loaded.")


Config loaded.


## 3. Data: load and pool HARD + ASTD + ArSAS with register labels

Reuses the same preprocessing as the ASTD/ArSAS replication notebook. HARD is
loaded fresh here (it wasn't part of that notebook). All three are pooled into
one training set with a register label: **0 = formal (HARD)**, **1 = informal
(ASTD, ArSAS)**.

In [5]:
class ArabicPreprocessor:
    @staticmethod
    def normalize_arabic(text):
        if not isinstance(text, str):
            return ""
        text = re.sub("[\u0625\u0623\u0622\u0627]", "\u0627", text)
        text = re.sub("\u0649", "\u064a", text)
        text = re.sub("\u0629", "\u0647", text)
        return re.sub("[\u0617-\u061A\u064B-\u0652\u0640]", "", text)

    @staticmethod
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = re.sub(r"http\S+|www\S+|https\S+", "", text)
        text = re.sub(r"\S+@\S+", "", text)
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def preprocess(text):
        return ArabicPreprocessor.clean_text(ArabicPreprocessor.normalize_arabic(text))


In [6]:
DATA_CACHE_DIR = f"{EL4ASA_ROOT}/data/replication_astd_arsas_prepared"
HARD_CACHE_PATH = f"{EL4ASA_ROOT}/data/hard_prepared.csv"
ASTD_CACHE_PATH = f"{DATA_CACHE_DIR}/astd_prepared.csv"
ARSAS_CACHE_PATH = f"{DATA_CACHE_DIR}/arsas_prepared.csv"


def prepare_hard():
    """Loads HARD directly from the file already present in your Drive project
    (EL4ASA/data/balanced-reviews.csv), rather than downloading -- this is the
    same balanced, 93,700-review file the rest of this project is built on,
    avoiding any ambiguity from re-deriving it from a different mirror."""
    source_path = f"{EL4ASA_ROOT}/data/balanced-reviews.csv"
    if not os.path.exists(source_path):
        raise FileNotFoundError(
            f"Expected to find the HARD source file at {source_path} (matching "
            f"the file already in your EL4ASA/data/ folder). If it lives "
            f"elsewhere, edit `source_path` above. Do not substitute a freshly "
            f"downloaded copy without checking it matches: the paper's Table 1 "
            f"reports 93,700 total reviews for the balanced version."
        )

    df = pd.read_csv(source_path)

    # Defensive column detection -- fail loudly and specifically rather than
    # silently mis-mapping columns if the schema differs from what's expected.
    text_col_candidates = [c for c in df.columns if c.lower() in ('review', 'text')]
    label_col_candidates = [c for c in df.columns if c.lower() in ('rating', 'label')]
    assert text_col_candidates, (
        f"Could not find a text column (expected 'review' or 'text'). "
        f"Actual columns: {df.columns.tolist()}"
    )
    assert label_col_candidates, (
        f"Could not find a rating/label column (expected 'rating' or 'label'). "
        f"Actual columns: {df.columns.tolist()}"
    )
    text_col, rating_col = text_col_candidates[0], label_col_candidates[0]
    print(f"Using columns: text='{text_col}', rating='{rating_col}'")

    df = df[[text_col, rating_col]].rename(columns={text_col: 'text', rating_col: 'rating'})
    print(f"Raw rating value distribution:\n{df['rating'].value_counts().sort_index()}")

    # The balanced-reviews file should already be pre-binarized to positive/negative
    # (no neutral); handle both a raw 1-5 rating column and an already-binary one.
    unique_ratings = sorted(df['rating'].dropna().unique().tolist())
    if set(unique_ratings).issubset({0, 1}):
        print("Rating column already binary (0/1) -- using directly as label.")
        df['label'] = df['rating'].astype(int)
    elif set(unique_ratings).issubset({1, 2, 4, 5}) or set(unique_ratings).issubset({1, 2, 3, 4, 5}):
        df = df[df['rating'] != 3].copy()
        df['label'] = (df['rating'] >= 4).astype(int)
    else:
        raise ValueError(
            f"Unexpected rating values: {unique_ratings}. Expected either binary "
            f"(0/1) or a 1-5 star scale. Inspect the source file's rating column "
            f"before proceeding."
        )

    df['text'] = df['text'].apply(ArabicPreprocessor.preprocess)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)

    print(f"Final: {len(df)} samples, {df['label'].mean()*100:.1f}% positive "
          f"(paper's Table 1 reports 93,700 total, ~50.4% positive -- verify this matches)")
    return df[['text', 'label']]


# --- Prepare HARD if not already cached ---
if os.path.exists(HARD_CACHE_PATH):
    print(f"HARD already prepared at {HARD_CACHE_PATH} -- skipping.")
else:
    print("Preparing HARD from your existing balanced-reviews.csv...")
    hard_prepared = prepare_hard()
    hard_prepared.to_csv(HARD_CACHE_PATH, index=False)
    print(f"Saved {len(hard_prepared)} samples to {HARD_CACHE_PATH}")

# --- ASTD/ArSAS must already be prepared by astd_arsas_replication.ipynb Section 3a ---
for name, path in [("ASTD", ASTD_CACHE_PATH), ("ArSAS", ARSAS_CACHE_PATH)]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} prepared file not found at {path}. Run Section 3a of "
            f"astd_arsas_replication.ipynb first to generate it."
        )

print("\nAll three datasets available.")


HARD already prepared at /content/drive/MyDrive/EL4ASA/data/hard_prepared.csv -- skipping.

All three datasets available.


In [7]:
hard_df = pd.read_csv(HARD_CACHE_PATH)
astd_df = pd.read_csv(ASTD_CACHE_PATH)
arsas_df = pd.read_csv(ARSAS_CACHE_PATH)

hard_df["register"] = 0   # formal
astd_df["register"] = 1   # informal
arsas_df["register"] = 1  # informal

hard_df["dataset"] = "hard"
astd_df["dataset"] = "astd"
arsas_df["dataset"] = "arsas"

DATASETS = {"hard": hard_df, "astd": astd_df, "arsas": arsas_df}
for name, df in DATASETS.items():
    print(f"{name:8s}: {len(df):6d} samples, {df["label"].mean()*100:.1f}% positive, register={df["register"].iloc[0]}")

POOLED_DF = pd.concat([hard_df, astd_df, arsas_df], ignore_index=True)
print(f"\nPooled total: {len(POOLED_DF)} samples")
print(f"Register distribution: {POOLED_DF["register"].value_counts().to_dict()}")


hard    : 105698 samples, 50.0% positive, register=0
astd    :   2419 samples, 32.1% positive, register=1
arsas   :  11784 samples, 37.3% positive, register=1

Pooled total: 119901 samples
Register distribution: {0: 105698, 1: 14203}


## 4. Dataset splitting and PyTorch Dataset class

Splits are stratified by **label** within each source dataset separately (not
across the pooled set), so HARD/ASTD/ArSAS each keep their own proper 80/10/10
train/val/test split -- exactly mirroring the protocol used everywhere else in
this project. The pooled train/val sets (used for backbone fine-tuning and RAAF
training) are the union of each dataset's own train/val split; test evaluation
is always done per-dataset separately.

In [8]:
def split_dataset(df, seed):
    train_val, test = train_test_split(
        df, test_size=TEST_RATIO, random_state=seed, stratify=df["label"]
    )
    val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train, val = train_test_split(
        train_val, test_size=val_ratio_adjusted, random_state=seed, stratify=train_val["label"]
    )
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def make_pooled_splits(seed, balance_by_register=True):
    """Split each dataset separately, then pool the train/val portions.

    balance_by_register=True (default, recommended): down-samples HARD's train
    split to match the combined size of ASTD+ArSAS's train splits before
    pooling, so Stage 1 fine-tuning does not disproportionately pull backbone
    representations toward the formal register merely because HARD is ~5x
    larger. This is directly motivated by Kumar et al. 2022 (ArXiv, 951
    citations), who show fine-tuning can distort pretrained features and hurt
    out-of-distribution performance specifically when the distribution shift
    is large -- exactly the formal-vs-informal shift RAAF is designed to
    handle. An earlier version of this notebook pooled naively (no balancing),
    which risked exactly this failure mode before the fusion module even had
    a chance to help. Set to False to reproduce the naive-pooling behavior for
    comparison/ablation purposes.
    """
    splits = {name: split_dataset(df, seed) for name, df in DATASETS.items()}

    train_parts = {name: s[0] for name, s in splits.items()}
    val_parts = {name: s[0].__class__() for name, s in splits.items()}  # placeholder, overwritten below
    val_parts = {name: s[1] for name, s in splits.items()}
    test_sets = {name: s[2] for name, s in splits.items()}

    if balance_by_register:
        informal_train_size = len(train_parts["astd"]) + len(train_parts["arsas"])
        if len(train_parts["hard"]) > informal_train_size:
            train_parts["hard"] = train_parts["hard"].sample(n=informal_train_size, random_state=seed).reset_index(drop=True)
        informal_val_size = len(val_parts["astd"]) + len(val_parts["arsas"])
        if len(val_parts["hard"]) > informal_val_size:
            val_parts["hard"] = val_parts["hard"].sample(n=informal_val_size, random_state=seed).reset_index(drop=True)
        print(f"Register-balanced pooling: HARD train down-sampled to {len(train_parts['hard'])} "
              f"(from full size) to match ASTD+ArSAS combined ({informal_train_size})")

    pooled_train = pd.concat(list(train_parts.values()), ignore_index=True)
    pooled_val = pd.concat(list(val_parts.values()), ignore_index=True)
    return pooled_train, pooled_val, test_sets


## 5. Stage 1: Joint backbone fine-tuning (checkpointed)

Fine-tunes each of the 4 backbones on the **pooled** HARD+ASTD+ArSAS training
data (standard binary sentiment classification -- register is not used in this
stage, only in Stage 2's fusion module). Each backbone's fine-tuned weights are
checkpointed to Drive; if a checkpoint already exists for a given
(model, seed), it is loaded instead of retrained.

In [9]:
class SentimentBackbone(nn.Module):
    """A backbone + classification head, used only for Stage 1 fine-tuning.
    After fine-tuning, we discard the head and keep only the backbone for
    frozen feature extraction in Stage 2."""
    def __init__(self, model_id, num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = out.last_hidden_state[:, 0, :]  # [CLS] token
        logits = self.classifier(cls_embedding)
        return logits, cls_embedding


def backbone_ckpt_path(model_name, seed):
    return f"{BACKBONE_CKPT_DIR}/{model_name}_pooled_seed{seed}.pt"


def train_backbone(model_name, model_id, train_df, val_df, seed, verbose=True):
    set_seed_fn(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = SentimentBackbone(model_id).to(DEVICE)

    train_ds = SimpleTextDataset(train_df["text"], train_df["label"], tokenizer, MAX_LENGTH)
    val_ds = SimpleTextDataset(val_df["text"], val_df["label"], tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * NUM_EPOCHS_BACKBONE
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
    )

    best_val_acc = 0.0
    start_time = time.time()
    for epoch in range(NUM_EPOCHS_BACKBONE):
        model.train()
        loop = tqdm(train_loader, desc=f"{model_name} seed={seed} epoch={epoch+1}", disable=not verbose)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            logits, _ = model(input_ids, attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            if verbose:
                loop.set_postfix(loss=loss.item())

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                logits, _ = model(input_ids, attention_mask)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc

    training_time = time.time() - start_time
    return model, tokenizer, best_val_acc, training_time


class SimpleTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label": torch.tensor(self.labels[idx], dtype=torch.long)}


def set_seed_fn(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def free_memory(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


In [10]:
def run_backbone_finetuning(seed):
    """Fine-tunes all 4 backbones on the pooled data for one seed, checkpointing
    each to Drive. Skips any backbone already checkpointed for this seed."""
    pooled_train, pooled_val, _ = make_pooled_splits(seed, balance_by_register=False)
    print(f"Pooled train: {len(pooled_train)}, pooled val: {len(pooled_val)}")

    for model_name, model_id in MODELS.items():
        ckpt_path = backbone_ckpt_path(model_name, seed)
        if os.path.exists(ckpt_path):
            print(f"  [{model_name}] seed={seed}: checkpoint exists, skipping.")
            continue
        print(f"  [{model_name}] seed={seed}: fine-tuning on pooled data...")
        model, tokenizer, val_acc, train_time = train_backbone(
            model_name, model_id, pooled_train, pooled_val, seed
        )
        torch.save({
            "model_state_dict": model.encoder.state_dict(),
            "val_acc": val_acc,
            "training_time": train_time,
        }, ckpt_path)
        print(f"  [{model_name}] seed={seed}: done, val_acc={val_acc:.4f}, saved to {ckpt_path}")
        free_memory(model, tokenizer)


## 6. Timing test (run before committing to full backbone fine-tuning)

Pooled training data is roughly HARD-sized (HARD dominates: ~85K pooled train
samples vs. ASTD+ArSAS's combined ~18K), so expect Stage 1 timing to be similar
to one HARD-scale run per backbone. This test trains CAMeLBERT on seed 42 for
real (all 3 epochs) and extrapolates the full Stage 1 cost (4 backbones x 5 seeds).

In [11]:
print("Running timing test: camelbert, seed=42, UNBALANCED pooled data (~3.8x larger than balanced run)...")
_pooled_train, _pooled_val, _ = make_pooled_splits(seed=42, balance_by_register=False)
print(f"Pooled train size: {len(_pooled_train)}")

_t0 = time.time()
_model, _tok, _val_acc, _train_time = train_backbone(
    "camelbert", MODELS["camelbert"], _pooled_train, _pooled_val, seed=42, verbose=True
)
_elapsed = time.time() - _t0
free_memory(_model, _tok)

print(f"\n--- Timing test result ---")
print(f"Time for camelbert x 1 seed x pooled data (3 epochs): {_elapsed/60:.1f} minutes")
print(f"Validation accuracy reached: {_val_acc*100:.1f}%")

_total_backbone_runs = len(MODELS) * len(SEEDS)
_est_stage1_hours = (_elapsed * _total_backbone_runs) / 3600
print(f"\nEstimated Stage 1 total (4 backbones x {len(SEEDS)} seeds): {_est_stage1_hours:.1f} hours")
print(f"\n--- Recommendation ---")
if _est_stage1_hours <= 3:
    print("Comfortably fits in a single session. Proceed with all 5 seeds.")
elif _est_stage1_hours <= 10:
    print(f"Will need multiple sessions (~{_est_stage1_hours/3:.0f} sittings of ~3h).")
    print("Fine given checkpointing -- stop and resume freely across sessions.")
else:
    print(f"{_est_stage1_hours:.0f} hours is substantial. Consider reducing SEEDS to")
    print("e.g. [42, 123, 456] (3 seeds) in Section 2, or running Stage 1 backbone-by-backbone")
    print("across separate sessions (each backbone is independently checkpointed).")


Running timing test: camelbert, seed=42, UNBALANCED pooled data (~3.8x larger than balanced run)...
Pooled train size: 95919


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

camelbert seed=42 epoch=1:   0%|          | 0/5995 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

camelbert seed=42 epoch=2:   0%|          | 0/5995 [00:00<?, ?it/s]

camelbert seed=42 epoch=3:   0%|          | 0/5995 [00:00<?, ?it/s]


--- Timing test result ---
Time for camelbert x 1 seed x pooled data (3 epochs): 27.4 minutes
Validation accuracy reached: 95.5%

Estimated Stage 1 total (4 backbones x 5 seeds): 9.1 hours

--- Recommendation ---
Will need multiple sessions (~3 sittings of ~3h).
Fine given checkpointing -- stop and resume freely across sessions.


## 7. Run Stage 1 for all seeds

In [12]:
for seed in SEEDS:
    print(f"\n{'='*70}\nSTAGE 1 -- SEED {seed}\n{'='*70}")
    run_backbone_finetuning(seed)

print("\n\nStage 1 complete for all requested seeds.")



STAGE 1 -- SEED 42
Pooled train: 95919, pooled val: 11991
  [arabert] seed=42: checkpoint exists, skipping.
  [marbert] seed=42: checkpoint exists, skipping.
  [xlm-roberta] seed=42: checkpoint exists, skipping.
  [camelbert] seed=42: checkpoint exists, skipping.

STAGE 1 -- SEED 123
Pooled train: 95919, pooled val: 11991
  [arabert] seed=123: checkpoint exists, skipping.
  [marbert] seed=123: checkpoint exists, skipping.
  [xlm-roberta] seed=123: checkpoint exists, skipping.
  [camelbert] seed=123: checkpoint exists, skipping.

STAGE 1 -- SEED 456
Pooled train: 95919, pooled val: 11991
  [arabert] seed=456: checkpoint exists, skipping.
  [marbert] seed=456: checkpoint exists, skipping.
  [xlm-roberta] seed=456: checkpoint exists, skipping.
  [camelbert] seed=456: checkpoint exists, skipping.

STAGE 1 -- SEED 789
Pooled train: 95919, pooled val: 11991
  [arabert] seed=789: checkpoint exists, skipping.
  [marbert] seed=789: checkpoint exists, skipping.
  [xlm-roberta] seed=789: checkpo

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


camelbert seed=2024 epoch=1:   0%|          | 0/5995 [00:00<?, ?it/s]

camelbert seed=2024 epoch=2:   0%|          | 0/5995 [00:00<?, ?it/s]

camelbert seed=2024 epoch=3:   0%|          | 0/5995 [00:00<?, ?it/s]

  [camelbert] seed=2024: done, val_acc=0.9568, saved to /content/drive/MyDrive/EL4ASA/raaf_unbalanced/backbones/camelbert_pooled_seed2024.pt


Stage 1 complete for all requested seeds.


## 8. Stage 2a: Extract and cache frozen embeddings

With backbones fine-tuned and frozen, extract [CLS] embeddings for the pooled
train/val sets and each dataset's own test set. Cached to Drive as `.npz` files
so Stage 2b (RAAF module training) can be iterated quickly without recomputing
embeddings every time.

In [13]:
def load_frozen_backbone(model_name, model_id, seed):
    ckpt_path = backbone_ckpt_path(model_name, seed)
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"No checkpoint at {ckpt_path} -- run Stage 1 for seed={seed} first.")
    encoder = AutoModel.from_pretrained(model_id)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    encoder.load_state_dict(ckpt["model_state_dict"])
    encoder.to(DEVICE)
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad = False
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    return encoder, tokenizer


def extract_embeddings(df, encoders, tokenizers, batch_size=32):
    """Returns a dict: model_name -> np.array of shape (n_samples, hidden_size)."""
    embeddings = {name: [] for name in MODEL_NAMES}
    n = len(df)
    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc="Extracting embeddings"):
            batch_texts = df["text"].iloc[start:start+batch_size].tolist()
            for name in MODEL_NAMES:
                enc = tokenizers[name](batch_texts, truncation=True, padding=True,
                                        max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
                out = encoders[name](**enc)
                cls_emb = out.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings[name].append(cls_emb)
    return {name: np.concatenate(v, axis=0) for name, v in embeddings.items()}


def embeddings_cache_path(split_name, seed):
    return f"{RAAF_DIR}/embeddings/{split_name}_seed{seed}.npz"


def extract_and_cache_all(seed):
    os.makedirs(f"{RAAF_DIR}/embeddings", exist_ok=True)
    pooled_train, pooled_val, test_sets = make_pooled_splits(seed, balance_by_register=False)

    encoders, tokenizers = {}, {}
    for name, model_id in MODELS.items():
        encoders[name], tokenizers[name] = load_frozen_backbone(name, model_id, seed)

    splits_to_process = {"pooled_train": pooled_train, "pooled_val": pooled_val}
    splits_to_process.update({f"test_{name}": df for name, df in test_sets.items()})

    for split_name, df in splits_to_process.items():
        cache_path = embeddings_cache_path(split_name, seed)
        if os.path.exists(cache_path):
            print(f"  [{split_name}] seed={seed}: cache exists, skipping.")
            continue
        print(f"  [{split_name}] seed={seed}: extracting embeddings for {len(df)} samples...")
        embs = extract_embeddings(df, encoders, tokenizers)
        save_dict = {f"emb_{name}": embs[name] for name in MODEL_NAMES}
        save_dict["label"] = df["label"].values
        save_dict["register"] = df["register"].values
        np.savez(cache_path, **save_dict)
        print(f"  [{split_name}] seed={seed}: cached to {cache_path}")

    free_memory(*encoders.values())


In [14]:
for seed in SEEDS:
    print(f"\n{'='*70}\nEXTRACTING EMBEDDINGS -- SEED {seed}\n{'='*70}")
    extract_and_cache_all(seed)

print("\n\nEmbedding extraction complete for all requested seeds.")



EXTRACTING EMBEDDINGS -- SEED 42


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  [test_hard] seed=2024: cached to /content/drive/MyDrive/EL4ASA/raaf_unbalanced/embeddings/test_hard_seed2024.npz
  [test_astd] seed=2024: extracting embeddings for 242 samples...


Extracting embeddings:   0%|          | 0/8 [00:00<?, ?it/s]

  [test_astd] seed=2024: cached to /content/drive/MyDrive/EL4ASA/raaf_unbalanced/embeddings/test_astd_seed2024.npz
  [test_arsas] seed=2024: extracting embeddings for 1179 samples...


Extracting embeddings:   0%|          | 0/37 [00:00<?, ?it/s]

  [test_arsas] seed=2024: cached to /content/drive/MyDrive/EL4ASA/raaf_unbalanced/embeddings/test_arsas_seed2024.npz


Embedding extraction complete for all requested seeds.


## 9. Stage 2b: The RAAF module itself

Operates on cached frozen embeddings (fast -- no backbone forward passes needed
here). Architecture:

1. Per-backbone linear projection to a common `FUSION_DIM`-dimensional space.
2. **Register predictor**: MLP on the mean of the 4 projected embeddings, predicting
   formal (0) vs. informal (1) register.
3. **Register-conditioned query**: the softmax register probabilities are used to
   softly index a learned register embedding table, producing a single query vector
   (differentiable end-to-end, not a hard argmax).
4. **Cross-attention fusion**: `nn.MultiheadAttention` with the register-conditioned
   vector as the query, and the 4 backbone embeddings as keys/values -- so fusion
   weighting is explicitly conditioned on predicted register.
5. Final classifier on `[fused_representation, register_query]` concatenated.

Trained with a joint loss: sentiment cross-entropy + `REGISTER_LOSS_WEIGHT` x
register cross-entropy (multi-task learning keeps the register signal well-calibrated
throughout training, not just at initialization).

In [15]:
class RAAFModule(nn.Module):
    def __init__(self, backbone_dim=768, fusion_dim=FUSION_DIM, num_heads=NUM_FUSION_HEADS,
                 num_register_labels=NUM_REGISTER_LABELS, num_sentiment_labels=NUM_SENTIMENT_LABELS):
        super().__init__()
        self.projections = nn.ModuleDict({
            name: nn.Linear(backbone_dim, fusion_dim) for name in MODEL_NAMES
        })
        self.register_predictor = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(fusion_dim // 2, num_register_labels),
        )
        self.register_embedding = nn.Embedding(num_register_labels, fusion_dim)
        self.fusion_attention = nn.MultiheadAttention(
            embed_dim=fusion_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(fusion_dim, num_sentiment_labels),
        )

    def forward(self, embeddings_dict, return_attention=False):
        projected = [self.projections[name](embeddings_dict[name]) for name in MODEL_NAMES]
        stacked = torch.stack(projected, dim=1)  # (batch, 4, fusion_dim)

        mean_emb = stacked.mean(dim=1)  # (batch, fusion_dim)
        register_logits = self.register_predictor(mean_emb)  # (batch, num_register_labels)
        register_probs = F.softmax(register_logits, dim=-1)
        register_query = register_probs @ self.register_embedding.weight  # (batch, fusion_dim)

        query = register_query.unsqueeze(1)  # (batch, 1, fusion_dim)
        fused, attn_weights = self.fusion_attention(query, stacked, stacked)
        fused = fused.squeeze(1)  # (batch, fusion_dim)

        combined = torch.cat([fused, register_query], dim=-1)
        sentiment_logits = self.classifier(combined)

        if return_attention:
            return sentiment_logits, register_logits, attn_weights.squeeze(1)  # (batch, 4) per-backbone weights
        return sentiment_logits, register_logits


In [16]:
class CachedEmbeddingDataset(Dataset):
    """Serves cached embeddings from an .npz file (fast, no tokenization/forward pass)."""
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.embeddings = {name: torch.tensor(data[f"emb_{name}"], dtype=torch.float32) for name in MODEL_NAMES}
        self.labels = torch.tensor(data["label"], dtype=torch.long)
        self.registers = torch.tensor(data["register"], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "embeddings": {name: self.embeddings[name][idx] for name in MODEL_NAMES},
            "label": self.labels[idx],
            "register": self.registers[idx],
        }


def cached_collate_fn(batch):
    return {
        "embeddings": {name: torch.stack([b["embeddings"][name] for b in batch]) for name in MODEL_NAMES},
        "label": torch.stack([b["label"] for b in batch]),
        "register": torch.stack([b["register"] for b in batch]),
    }


## 10. Train and evaluate RAAF (checkpointed per seed)

In [17]:
def raaf_ckpt_path(seed):
    return f"{RAAF_RESULTS_DIR}/raaf_model_seed{seed}.pt"

def raaf_test_results_path(dataset_name, seed):
    return f"{RAAF_RESULTS_DIR}/raaf_test_{dataset_name}_seed{seed}.json"


def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


def train_raaf(seed, verbose=True):
    ckpt_path = raaf_ckpt_path(seed)
    train_cache = embeddings_cache_path("pooled_train", seed)
    val_cache = embeddings_cache_path("pooled_val", seed)

    train_ds = CachedEmbeddingDataset(train_cache)
    val_ds = CachedEmbeddingDataset(val_cache)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=cached_collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=cached_collate_fn)

    set_seed_fn(seed)
    model = RAAFModule().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=RAAF_LEARNING_RATE, weight_decay=0.01)

    best_val_acc = 0.0
    best_state = None
    for epoch in range(NUM_EPOCHS_RAAF):
        model.train()
        loop = tqdm(train_loader, desc=f"RAAF seed={seed} epoch={epoch+1}", disable=not verbose)
        for batch in loop:
            optimizer.zero_grad()
            embeddings = {name: batch["embeddings"][name].to(DEVICE) for name in MODEL_NAMES}
            labels = batch["label"].to(DEVICE)
            registers = batch["register"].to(DEVICE)

            sentiment_logits, register_logits = model(embeddings)
            loss = (F.cross_entropy(sentiment_logits, labels)
                    + REGISTER_LOSS_WEIGHT * F.cross_entropy(register_logits, registers))
            loss.backward()
            optimizer.step()
            if verbose:
                loop.set_postfix(loss=loss.item())

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                embeddings = {name: batch["embeddings"][name].to(DEVICE) for name in MODEL_NAMES}
                labels = batch["label"].to(DEVICE)
                sentiment_logits, _ = model(embeddings)
                preds = torch.argmax(sentiment_logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_acc = accuracy_score(val_labels, val_preds)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)
    print(f"RAAF seed={seed}: best val_acc={best_val_acc:.4f}, saved to {ckpt_path}")
    return model


def evaluate_raaf_on_dataset(model, dataset_name, seed):
    test_cache = embeddings_cache_path(f"test_{dataset_name}", seed)
    test_ds = CachedEmbeddingDataset(test_cache)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=cached_collate_fn)

    model.eval()
    preds, labels_list, register_preds, register_labels_list, attn_weights_list = [], [], [], [], []
    with torch.no_grad():
        for batch in test_loader:
            embeddings = {name: batch["embeddings"][name].to(DEVICE) for name in MODEL_NAMES}
            labels = batch["label"].to(DEVICE)
            registers = batch["register"].to(DEVICE)
            sentiment_logits, register_logits, attn_weights = model(embeddings, return_attention=True)
            preds.extend(torch.argmax(sentiment_logits, dim=1).cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            register_preds.extend(torch.argmax(register_logits, dim=1).cpu().numpy())
            register_labels_list.extend(registers.cpu().numpy())
            attn_weights_list.append(attn_weights.cpu().numpy())

    metrics = calculate_metrics(labels_list, preds)
    register_acc = accuracy_score(register_labels_list, register_preds)
    mean_attn = np.concatenate(attn_weights_list, axis=0).mean(axis=0)  # mean attention weight per backbone

    result = {
        **metrics,
        "register_prediction_accuracy": register_acc,
        "mean_attention_weights": {name: float(w) for name, w in zip(MODEL_NAMES, mean_attn)},
    }
    return result


In [18]:
for seed in SEEDS:
    ckpt_path = raaf_ckpt_path(seed)
    if os.path.exists(ckpt_path):
        print(f"RAAF seed={seed}: checkpoint exists, loading instead of retraining.")
        model = RAAFModule().to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    else:
        model = train_raaf(seed)

    for dataset_name in DATASETS:
        result_path = raaf_test_results_path(dataset_name, seed)
        if os.path.exists(result_path):
            print(f"  [{dataset_name}] seed={seed}: result exists, skipping.")
            continue
        result = evaluate_raaf_on_dataset(model, dataset_name, seed)
        with open(result_path, "w") as f:
            json.dump(result, f, indent=2)
        print(f"  [{dataset_name}] seed={seed}: F1={result["f1"]*100:.2f}%, "
              f"register_acc={result["register_prediction_accuracy"]*100:.1f}%, "
              f"attn={result["mean_attention_weights"]}")

    free_memory(model)

print("\n\nRAAF training and evaluation complete for all requested seeds.")


RAAF seed=42 epoch=1:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=42 epoch=2:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=42 epoch=3:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=42 epoch=4:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=42 epoch=5:   0%|          | 0/1499 [00:00<?, ?it/s]

...

RAAF seed=2024 epoch=7:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=2024 epoch=8:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=2024 epoch=9:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=2024 epoch=10:   0%|          | 0/1499 [00:00<?, ?it/s]

RAAF seed=2024: best val_acc=0.9559, saved to /content/drive/MyDrive/EL4ASA/raaf_unbalanced/results/raaf_model_seed2024.pt
  [hard] seed=2024: F1=96.16%, register_acc=100.0%, attn={'arabert': 0.3760353624820709, 'marbert': 0.302223265171051, 'xlm-roberta': 0.13347795605659485, 'camelbert': 0.188263401389122}
  [astd] seed=2024: F1=85.57%, register_acc=99.6%, attn={'arabert': 0.35540708899497986, 'marbert': 0.08264462649822235, 'xlm-roberta': 0.08264463394880295, 'camelbert': 0.47930365800857544}
  [arsas] seed=2024: F1=89.94%, register_acc=100.0%, attn={'arabert': 0.33979931473731995, 'marbert': 0.09117896854877472, 'xlm-roberta': 0.09349758177995682, 'camelbert': 0.4755241274833679}


RAAF training and evaluation complete for all requested seeds.


## 11. Compare RAAF against existing baselines (statistical testing)

Compares RAAF's per-dataset F1 (mean $\pm$ std across seeds) against the best
single model and best ensemble already established for that dataset, and runs
the same paired t-test used throughout the rest of this project. **The key
question**: does RAAF achieve significance on ASTD/ArSAS where plain stacking
did not (p=0.982 ASTD, p=0.071 ArSAS)?

In [19]:
from scipy import stats

# Existing results already established in the manuscript (mean F1 % per seed
# is not directly available here for HARD/ASTD/ArSAS individual/ensemble
# baselines beyond mean+-std -- for a fully rigorous paired test against RAAF's
# per-seed numbers, use the per-seed baseline F1 values from your saved results
# JSONs (results/ for HARD, replication_astd_arsas/results/ for ASTD/ArSAS)
# rather than the hardcoded means below where per-seed precision matters.

EXISTING_BEST_INDIVIDUAL = {
    "hard": ("camelbert", 96.24, 0.19),
    "astd": ("marbert", 87.62, 1.19),
    "arsas": ("marbert", 91.24, 0.32),
}
EXISTING_BEST_ENSEMBLE = {
    "hard": ("stacking", 96.40, 0.14),
    "astd": ("soft_voting", 87.60, 1.28),
    "arsas": ("soft_voting", 91.59, 0.14),
}

def collect_raaf_f1s(dataset_name):
    f1s = []
    for seed in SEEDS:
        path = raaf_test_results_path(dataset_name, seed)
        if os.path.exists(path):
            with open(path) as f:
                f1s.append(json.load(f)["f1"] * 100)
    return np.array(f1s)


print(f"{'Dataset':8s} {'RAAF':>18s} {'Best Individual':>22s} {'Best Ensemble':>20s}")
raaf_summary = {}
for dataset_name in DATASETS:
    raaf_f1s = collect_raaf_f1s(dataset_name)
    if len(raaf_f1s) == 0:
        print(f"{dataset_name:8s}  (no RAAF results yet)")
        continue
    raaf_mean, raaf_std = raaf_f1s.mean(), raaf_f1s.std()
    bi_name, bi_mean, bi_std = EXISTING_BEST_INDIVIDUAL[dataset_name]
    be_name, be_mean, be_std = EXISTING_BEST_ENSEMBLE[dataset_name]
    print(f"{dataset_name:8s} {raaf_mean:6.2f}+-{raaf_std:4.2f} (n={len(raaf_f1s)})"
          f"   {bi_name}={bi_mean:6.2f}+-{bi_std:4.2f}"
          f"   {be_name}={be_mean:6.2f}+-{be_std:4.2f}")
    raaf_summary[dataset_name] = {"mean": raaf_mean, "std": raaf_std, "n": len(raaf_f1s), "f1s": raaf_f1s.tolist()}

print("\nNote: significance testing against the best-individual/best-ensemble baselines")
print("requires their per-seed F1 arrays (not just mean+-std) for a valid paired t-test.")
print("Load those from your saved results JSONs below if you want an exact paired test")
print("(they use the same 5 seeds, so pairing is valid) rather than an approximate")
print("two-sample comparison from summary statistics alone.")


Dataset                RAAF        Best Individual        Best Ensemble
hard      96.20+-0.07 (n=5)   camelbert= 96.24+-0.19   stacking= 96.40+-0.14
astd      85.94+-0.73 (n=5)   marbert= 87.62+-1.19   soft_voting= 87.60+-1.28
arsas     90.63+-0.74 (n=5)   marbert= 91.24+-0.32   soft_voting= 91.59+-0.14

Note: significance testing against the best-individual/best-ensemble baselines
requires their per-seed F1 arrays (not just mean+-std) for a valid paired t-test.
Load those from your saved results JSONs below if you want an exact paired test
(they use the same 5 seeds, so pairing is valid) rather than an approximate
two-sample comparison from summary statistics alone.


In [20]:
# Optional: exact paired test against HARD baselines, if you have the per-seed
# JSONs from the original experiment available in this Colab session (upload or
# mount the same results/ folder used in the original HARD experiment).
#
# Example for HARD (adjust paths to wherever those per-seed JSONs are accessible):
#
# camelbert_f1s = []
# for seed in SEEDS:
#     with open(f"/path/to/results/camelbert_results_seed{seed}.json") as f:
#         camelbert_f1s.append(json.load(f)["f1"] * 100)
# camelbert_f1s = np.array(camelbert_f1s)
# raaf_hard_f1s = collect_raaf_f1s("hard")
# t_stat, p_val = stats.ttest_rel(raaf_hard_f1s, camelbert_f1s)
# print(f"RAAF vs CAMeLBERT on HARD: t={t_stat:.3f}, p={p_val:.4f}")
#
# Repeat similarly for ASTD/ArSAS using the replication notebook's per-seed
# result JSONs (astd_marbert_seed{seed}.json, etc.) and the ensemble JSONs
# (astd_ensembles_seed{seed}.json -> ["soft_voting"]["f1"]) for the ensemble comparison.

print("See the commented template above -- fill in your actual per-seed JSON paths")
print("to run the exact paired significance test against each baseline.")


See the commented template above -- fill in your actual per-seed JSON paths
to run the exact paired significance test against each baseline.


## 12. Interpretability check: did RAAF learn the hypothesized routing pattern?

The central hypothesis motivating RAAF's design: attention should shift toward
MARBERT for informal-register inputs and toward CAMeLBERT for formal-register
inputs. The `mean_attention_weights` saved in each test result lets us check
this directly, rather than just trusting the architecture "should" behave this way.

In [21]:
print(f"{'Dataset':8s} {'Register':10s} " + " ".join(f"{m:>12s}" for m in MODEL_NAMES))
for dataset_name in DATASETS:
    for seed in SEEDS:
        path = raaf_test_results_path(dataset_name, seed)
        if not os.path.exists(path):
            continue
        with open(path) as f:
            r = json.load(f)
        register = "formal" if dataset_name == "hard" else "informal"
        weights = r["mean_attention_weights"]
        print(f"{dataset_name:8s} {register:10s} " + " ".join(f"{weights[m]:12.3f}" for m in MODEL_NAMES))
        break  # one representative seed is enough for this qualitative check

print("\nIf RAAF learned the hypothesized pattern: camelbert's attention weight")
print("should be highest for 'hard' (formal), and marbert's should be highest")
print("for 'astd'/'arsas' (informal). If this does NOT hold, RAAF's mechanism")
print("is not doing what it was designed to do, even if raw accuracy looks fine --")
print("report this honestly either way, it is a real finding about the architecture.")


Dataset  Register        arabert      marbert  xlm-roberta    camelbert
hard     formal            0.432        0.373        0.118        0.077
astd     informal          0.360        0.329        0.112        0.199
arsas    informal          0.349        0.348        0.114        0.188

If RAAF learned the hypothesized pattern: camelbert's attention weight
should be highest for 'hard' (formal), and marbert's should be highest
for 'astd'/'arsas' (informal). If this does NOT hold, RAAF's mechanism
is not doing what it was designed to do, even if raw accuracy looks fine --
report this honestly either way, it is a real finding about the architecture.


## 13. LaTeX table row generator

In [22]:
print("RAAF row for the manuscript's cross-dataset table:\n")
row = ["RAAF"]
for dataset_name in ["hard", "astd", "arsas"]:
    f1s = collect_raaf_f1s(dataset_name)
    if len(f1s) > 0:
        row.append(f"{f1s.mean():.2f} $\\pm$ {f1s.std():.2f}")
    else:
        row.append("[TBD]")
print(f"{row[0]} & {row[1]} & {row[2]} & {row[3]} \\\\")


RAAF row for the manuscript's cross-dataset table:

RAAF & 96.20 $\pm$ 0.07 & 85.94 $\pm$ 0.73 & 90.63 $\pm$ 0.74 \\


## 14. Testing the theoretical predictions (P1, P2, P3)

Section~9's theoretical account (Ben-David et al. domain adaptation bound +
Jacobs/Jordan mixture-of-experts theory) generates three falsifiable predictions
beyond the headline accuracy comparison. This section computes each directly.

**P1 (confidence-gain correlation)**: examples where the register predictor is
more confident should show larger RAAF gains over a register-blind naive baseline.

**P2 (attention-divergence correlation)**: RAAF's learned attention weights should
correlate with an *independently computed* domain-divergence proxy (embedding
distance to per-backbone formal/informal centroids) -- not derived from RAAF's
own register predictor, to avoid circularity.

**P3 (predicted failure mode)**: RAAF should show elevated error specifically on
code-switched/register-ambiguous text, where register is a poor proxy for the
true source of backbone disagreement.

In [23]:
def raaf_detailed_results_path(dataset_name, seed):
    return f"{RAAF_RESULTS_DIR}/raaf_detailed_{dataset_name}_seed{seed}.npz"


def evaluate_raaf_detailed(model, dataset_name, seed):
    """Extended evaluation: saves per-example predictions, register confidence,
    and attention weights (not just aggregate metrics) for the P1/P2/P3 tests."""
    test_cache = embeddings_cache_path(f"test_{dataset_name}", seed)
    test_ds = CachedEmbeddingDataset(test_cache)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=cached_collate_fn, shuffle=False)

    model.eval()
    all_preds, all_labels, all_register_conf, all_attn = [], [], [], []
    all_backbone_preds = {name: [] for name in MODEL_NAMES}  # for the naive baseline
    with torch.no_grad():
        for batch in test_loader:
            embeddings = {name: batch["embeddings"][name].to(DEVICE) for name in MODEL_NAMES}
            labels = batch["label"].to(DEVICE)
            sentiment_logits, register_logits, attn_weights = model(embeddings, return_attention=True)

            all_preds.append(torch.argmax(sentiment_logits, dim=1).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            register_conf = torch.softmax(register_logits, dim=-1).max(dim=-1).values
            all_register_conf.append(register_conf.cpu().numpy())
            all_attn.append(attn_weights.cpu().numpy())

            # Naive register-blind baseline: simple linear classifier per backbone
            # approximated here by the sign of each backbone's own projected
            # embedding through RAAF's classifier applied to a uniform (not
            # register-conditioned) average -- i.e. bypass the learned query
            # entirely and use an unweighted mean of the four backbones' fused
            # contributions as the "register-blind" comparison point.
            with torch.no_grad():
                projected = [model.projections[name](embeddings[name]) for name in MODEL_NAMES]
                stacked = torch.stack(projected, dim=1)
                naive_fused = stacked.mean(dim=1)
                naive_register_query = torch.zeros_like(naive_fused)  # no register conditioning
                naive_combined = torch.cat([naive_fused, naive_register_query], dim=-1)
                naive_logits = model.classifier(naive_combined)
                naive_preds = torch.argmax(naive_logits, dim=1)
            all_backbone_preds["naive"] = all_backbone_preds.get("naive", [])
            all_backbone_preds["naive"].append(naive_preds.cpu().numpy())

    result = {
        "preds": np.concatenate(all_preds),
        "labels": np.concatenate(all_labels),
        "register_confidence": np.concatenate(all_register_conf),
        "attention_weights": np.concatenate(all_attn, axis=0),  # (n, 4)
        "naive_preds": np.concatenate(all_backbone_preds["naive"]),
    }
    np.savez(raaf_detailed_results_path(dataset_name, seed), **result)
    return result


In [24]:
# Run detailed evaluation for all seeds/datasets (skips if already saved)
for seed in SEEDS:
    ckpt_path = raaf_ckpt_path(seed)
    if not os.path.exists(ckpt_path):
        print(f"No RAAF checkpoint for seed={seed} yet -- skipping detailed eval.")
        continue
    model = RAAFModule().to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    for dataset_name in DATASETS:
        detail_path = raaf_detailed_results_path(dataset_name, seed)
        if os.path.exists(detail_path):
            print(f"  [{dataset_name}] seed={seed}: detailed results exist, skipping.")
            continue
        evaluate_raaf_detailed(model, dataset_name, seed)
        print(f"  [{dataset_name}] seed={seed}: detailed results saved.")
    free_memory(model)


  [hard] seed=42: detailed results saved.
  [astd] seed=42: detailed results saved.
  [arsas] seed=42: detailed results saved.
  [hard] seed=123: detailed results saved.
  [astd] seed=123: detailed results saved.
  [arsas] seed=123: detailed results saved.
  [hard] seed=456: detailed results saved.
  [astd] seed=456: detailed results saved.
  [arsas] seed=456: detailed results saved.
  [hard] seed=789: detailed results saved.
  [astd] seed=789: detailed results saved.
  [arsas] seed=789: detailed results saved.
  [hard] seed=2024: detailed results saved.
  [astd] seed=2024: detailed results saved.
  [arsas] seed=2024: detailed results saved.


### P1: Register-confidence vs. fusion-gain correlation

In [25]:
def test_p1(dataset_name, seed):
    path = raaf_detailed_results_path(dataset_name, seed)
    if not os.path.exists(path):
        return None
    d = np.load(path)
    raaf_correct = (d["preds"] == d["labels"]).astype(int)
    naive_correct = (d["naive_preds"] == d["labels"]).astype(int)
    gain = raaf_correct - naive_correct  # +1: RAAF right, naive wrong. -1: reverse. 0: same.
    confidence = d["register_confidence"]

    rho, p_val = stats.spearmanr(confidence, gain)
    median_conf = np.median(confidence)
    mean_gain_high = gain[confidence > median_conf].mean()
    mean_gain_low = gain[confidence <= median_conf].mean()
    return {"rho": rho, "p_value": p_val, "n": len(gain),
            "mean_gain_high_conf": mean_gain_high,
            "mean_gain_low_conf": mean_gain_low}


print("=== P1: Register-confidence vs. fusion-gain correlation ===\n")
for dataset_name in DATASETS:
    for seed in SEEDS:
        result = test_p1(dataset_name, seed)
        if result is None:
            continue
        rho = result["rho"]
        p_value = result["p_value"]
        gain_high = result["mean_gain_high_conf"]
        gain_low = result["mean_gain_low_conf"]
        print(f"{dataset_name:8s} seed={seed}: rho={rho:.3f} (p={p_value:.4f}), "
              f"mean gain (high-conf examples)={gain_high:+.3f}, "
              f"mean gain (low-conf examples)={gain_low:+.3f}")
        break  # one seed is enough to illustrate; aggregate properly before reporting in the paper

print("\nP1 supported if: rho > 0 and significant, AND mean gain is higher for high-confidence examples.")
print("This would mean RAAF helps more precisely where its register signal is more reliable --")
print("consistent with the theoretical account, not just a general accuracy bump.")


=== P1: Register-confidence vs. fusion-gain correlation ===

hard     seed=42: rho=-0.584 (p=0.0000), mean gain (high-conf examples)=+nan, mean gain (low-conf examples)=+0.419
astd     seed=42: rho=-0.145 (p=0.0243), mean gain (high-conf examples)=+0.636, mean gain (low-conf examples)=+0.653
arsas    seed=42: rho=-0.475 (p=0.0000), mean gain (high-conf examples)=+0.344, mean gain (low-conf examples)=+0.793

P1 supported if: rho > 0 and significant, AND mean gain is higher for high-confidence examples.
This would mean RAAF helps more precisely where its register signal is more reliable --
consistent with the theoretical account, not just a general accuracy bump.


### P2: Attention weights vs. independent domain-divergence proxy

Computed entirely independently of RAAF's own register predictor -- using only
raw embedding-space distances to per-backbone formal/informal centroids -- to
avoid circularity (testing the gate's attention against a proxy derived from
the gate itself would not be a real test).

In [26]:
def compute_register_centroids(seed):
    """Per-backbone centroids of formal (HARD) vs informal (ASTD+ArSAS) embeddings,
    computed from the pooled training set -- independent of RAAF's register predictor."""
    train_cache = embeddings_cache_path("pooled_train", seed)
    data = np.load(train_cache)
    registers = data["register"]
    centroids = {}
    for name in MODEL_NAMES:
        emb = data[f"emb_{name}"]
        centroids[name] = {
            "formal": emb[registers == 0].mean(axis=0),
            "informal": emb[registers == 1].mean(axis=0),
        }
    return centroids


def test_p2(dataset_name, seed):
    centroids = compute_register_centroids(seed)
    test_cache = embeddings_cache_path(f"test_{dataset_name}", seed)
    test_data = np.load(test_cache)
    detail_path = raaf_detailed_results_path(dataset_name, seed)
    if not os.path.exists(detail_path):
        return None
    detail = np.load(detail_path)
    attn = detail["attention_weights"]  # (n, 4)

    correlations = {}
    for i, name in enumerate(MODEL_NAMES):
        emb = test_data[f"emb_{name}"]
        dist_formal = np.linalg.norm(emb - centroids[name]["formal"], axis=1)
        dist_informal = np.linalg.norm(emb - centroids[name]["informal"], axis=1)
        informal_affinity = dist_formal - dist_informal  # higher = more informal-like
        rho, p_val = stats.spearmanr(informal_affinity, attn[:, i])
        correlations[name] = {"rho": rho, "p_value": p_val}
    return correlations


print("=== P2: Attention weight vs. independent domain-divergence proxy ===\n")
print("Expected if P2 holds: MARBERT's attention weight should correlate POSITIVELY")
print("with informal-affinity (gets more attention on informal-looking text); CAMeLBERT")
print("should correlate NEGATIVELY (gets more attention on formal-looking text).\n")

for dataset_name in DATASETS:
    for seed in SEEDS:
        result = test_p2(dataset_name, seed)
        if result is None:
            continue
        print(f"{dataset_name} (seed={seed}):")
        for name in MODEL_NAMES:
            rho = result[name]["rho"]
            p_value = result[name]["p_value"]
            print(f"  {name:12s}: rho={rho:+.3f} (p={p_value:.4f})")
        break


=== P2: Attention weight vs. independent domain-divergence proxy ===

Expected if P2 holds: MARBERT's attention weight should correlate POSITIVELY
with informal-affinity (gets more attention on informal-looking text); CAMeLBERT
should correlate NEGATIVELY (gets more attention on formal-looking text).

hard (seed=42):
  arabert     : rho=-0.061 (p=0.0000)
  marbert     : rho=-0.833 (p=0.0000)
  xlm-roberta : rho=+0.759 (p=0.0000)
  camelbert   : rho=-0.216 (p=0.0000)
astd (seed=42):
  arabert     : rho=-0.740 (p=0.0000)
  marbert     : rho=+0.663 (p=0.0000)
  xlm-roberta : rho=+0.391 (p=0.0000)
  camelbert   : rho=-0.339 (p=0.0000)
arsas (seed=42):
  arabert     : rho=-0.800 (p=0.0000)
  marbert     : rho=+0.712 (p=0.0000)
  xlm-roberta : rho=+0.549 (p=0.0000)
  camelbert   : rho=-0.326 (p=0.0000)


### P3: Predicted failure mode -- elevated error on code-switched/ambiguous text

Reuses the same code-switching detector from `linguistic_marker_analysis.ipynb`
(Latin-script sequences embedded in Arabic text) as a cheap, objective proxy for
"register is ambiguous / not well-characterized by a formal-vs-informal binary."
Theory (Section~9, P3) predicts RAAF's error rate should be elevated specifically
on this subset, since register alone does not fully capture why a backbone might
err on code-switched input.

In [27]:
import re as _re
LATIN_PATTERN = _re.compile(r"[A-Za-z]{2,}")

def has_code_switching(text):
    return bool(LATIN_PATTERN.search(str(text)))


def test_p3(dataset_name, seed):
    detail_path = raaf_detailed_results_path(dataset_name, seed)
    if not os.path.exists(detail_path):
        return None
    detail = np.load(detail_path)

    # Need the original text to check for code-switching -- re-derive the same
    # test split used for this seed/dataset (deterministic given the same seed).
    _, _, test_sets = make_pooled_splits(seed, balance_by_register=False)
    test_df = test_sets[dataset_name].reset_index(drop=True)

    assert len(test_df) == len(detail["preds"]), (
        f"Length mismatch: test_df has {len(test_df)} rows but detailed results "
        f"have {len(detail['preds'])} -- the split may not be reproducing "
        f"identically. Check that make_pooled_splits(seed, balance_by_register=False) is deterministic."
    )

    is_code_switched = test_df["text"].apply(has_code_switching).values
    correct = (detail["preds"] == detail["labels"]).astype(int)

    error_rate_cs = 1 - correct[is_code_switched].mean() if is_code_switched.sum() > 0 else float("nan")
    error_rate_non_cs = 1 - correct[~is_code_switched].mean()
    n_cs = is_code_switched.sum()

    if n_cs >= 5:  # only run the test if there are enough code-switched examples
        contingency = np.array([
            [correct[is_code_switched].sum(), (1 - correct[is_code_switched]).sum()],
            [correct[~is_code_switched].sum(), (1 - correct[~is_code_switched]).sum()],
        ])
        chi2, p_val, _, _ = stats.chi2_contingency(contingency)
    else:
        p_val = float("nan")

    return {"error_rate_code_switched": error_rate_cs, "error_rate_other": error_rate_non_cs,
            "n_code_switched": int(n_cs), "p_value": p_val}


print("=== P3: Error rate on code-switched vs. other text ===\n")
for dataset_name in DATASETS:
    for seed in SEEDS:
        result = test_p3(dataset_name, seed)
        if result is None:
            continue
        err_cs = result["error_rate_code_switched"]
        err_other = result["error_rate_other"]
        n_cs = result["n_code_switched"]
        p_value = result["p_value"]
        print(f"{dataset_name:8s} seed={seed}: error rate code-switched (n={n_cs})={err_cs:.3f}, "
              f"error rate other={err_other:.3f}, chi2 p={p_value:.4f}")
        break

print("\nP3 supported if: error rate is meaningfully and significantly higher on")
print("code-switched text. Note n may be small (code-switching is rare in these")
print("datasets per the linguistic-marker analysis) -- interpret p-values accordingly.")


=== P3: Error rate on code-switched vs. other text ===

hard     seed=42: error rate code-switched (n=244)=0.074, error rate other=0.037, chi2 p=0.0058
astd     seed=42: error rate code-switched (n=1)=0.000, error rate other=0.124, chi2 p=nan
arsas    seed=42: error rate code-switched (n=81)=0.099, error rate other=0.094, chi2 p=1.0000

P3 supported if: error rate is meaningfully and significantly higher on
code-switched text. Note n may be small (code-switching is rare in these
datasets per the linguistic-marker analysis) -- interpret p-values accordingly.


## 15. Final three-way comparison: unbalanced RAAF vs. balanced RAAF vs. baselines

This is the cell that resolves the confound. It reads the balanced run's results
directly from the original `raaf/` directory on Drive (read-only) alongside this
ablation's `raaf_unbalanced/` results.

In [28]:
BALANCED_RESULTS_DIR = f"{EL4ASA_ROOT}/raaf/results"   # original balanced run, read-only

def collect_f1s(results_dir, dataset_name):
    f1s = []
    for seed in SEEDS:
        path = f"{results_dir}/raaf_test_{dataset_name}_seed{seed}.json"
        if os.path.exists(path):
            with open(path) as f:
                f1s.append(json.load(f)["f1"] * 100)
    return np.array(f1s)

BASELINES = {
    "hard":  {"best_individual": ("CAMeLBERT", 96.24), "best_ensemble": ("stacking", 96.40)},
    "astd":  {"best_individual": ("MARBERT", 87.62), "best_ensemble": ("soft_voting", 87.60)},
    "arsas": {"best_individual": ("MARBERT", 91.24), "best_ensemble": ("soft_voting", 91.59)},
}

print(f"{'Dataset':8s} {'Unbalanced RAAF':>18s} {'Balanced RAAF':>16s} {'Best Indiv.':>16s} {'Best Ensemble':>16s}")
for ds in ["hard", "astd", "arsas"]:
    unbal = collect_f1s(RAAF_RESULTS_DIR, ds)
    bal = collect_f1s(BALANCED_RESULTS_DIR, ds)
    bi_name, bi_mean = BASELINES[ds]["best_individual"]
    be_name, be_mean = BASELINES[ds]["best_ensemble"]
    unbal_str = f"{unbal.mean():.2f}+-{unbal.std():.2f}" if len(unbal) else "(none)"
    bal_str = f"{bal.mean():.2f}+-{bal.std():.2f}" if len(bal) else "(none)"
    print(f"{ds:8s} {unbal_str:>18s} {bal_str:>16s} {bi_mean:>10.2f} ({bi_name[:4]}) {be_mean:>9.2f} ({be_name[:5]})")

print()
print("Interpretation:")
print(" - Unbalanced >> Balanced and roughly = baselines: the shortfall was the")
print("   undertraining confound; balanced pooling was too aggressive; the fusion")
print("   mechanism is not refuted on accuracy grounds.")
print(" - Unbalanced still clearly below baselines: the confound was not the main")
print("   cause; evidence turns against the fusion mechanism adding accuracy value.")
print(" - Also rerun the P1/P2/P3 sections above on this run and compare mechanism-level")
print("   results across the two pooling regimes before writing anything into the paper.")


Dataset     Unbalanced RAAF    Balanced RAAF      Best Indiv.    Best Ensemble
hard            96.20+-0.07      95.60+-0.14      96.24 (CAMe)     96.40 (stack)
astd            85.94+-0.73      87.09+-1.35      87.62 (MARB)     87.60 (soft_)
arsas           90.63+-0.74      90.85+-0.69      91.24 (MARB)     91.59 (soft_)

Interpretation:
 - Unbalanced >> Balanced and roughly = baselines: the shortfall was the
   undertraining confound; balanced pooling was too aggressive; the fusion
   mechanism is not refuted on accuracy grounds.
 - Unbalanced still clearly below baselines: the confound was not the main
   cause; evidence turns against the fusion mechanism adding accuracy value.
 - Also rerun the P1/P2/P3 sections above on this run and compare mechanism-level
   results across the two pooling regimes before writing anything into the paper.
